# Solution 03 — the recipe extractor

A `Recipe` model, an agent with `retries=3`, and one output validator that enforces the rules with
`ModelRetry`. Swap in any recipe text you like.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent, ModelRetry


class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    steps: list[str]
    minutes: int


recipe_agent = Agent(
    MODEL,
    output_type=Recipe,
    instructions="Extract the recipe from the user text. Invent nothing. `minutes` is the total time.",
    retries=3,
)


@recipe_agent.output_validator
def sanity_check(output: Recipe) -> Recipe:
    """Rules about meaning that the shape alone can't check."""
    if not output.ingredients:
        raise ModelRetry("The recipe has no ingredients. List every ingredient in the text.")
    if output.minutes <= 0:
        raise ModelRetry(
            f"{output.minutes} minutes is impossible. Use the total time from the text."
        )
    # Stretch: a friendlier rule that tells the model what to do instead.
    if len(output.ingredients) > 10:
        raise ModelRetry("More than 10 ingredients. Merge them into at most 10 core ingredients.")
    return output


sample = """
French toast: whisk 2 eggs with 100ml milk and a pinch of cinnamon.
Soak 2 slices of brioche, fry in butter about 3 minutes per side. Takes 10 minutes total.
"""

result = await recipe_agent.run(sample)
print(result.output)

The validator is a plain function, so you can check its "no" path without calling a model at all.

In [ ]:
recipe = result.output

rejected = False
try:
    sanity_check(Recipe(title="Air", ingredients=[], steps=["Breathe in."], minutes=1))
except ModelRetry:
    rejected = True

assert rejected, "a recipe with no ingredients must be rejected"
assert isinstance(recipe, Recipe)
assert 0 < len(recipe.ingredients) <= 10, recipe.ingredients
assert recipe.minutes > 0, recipe.minutes

**Notes**

- The validator runs *after* Pydantic has checked the shape. It is your last word on meaning.
- Each `ModelRetry` costs one more model request. A rule the model can never satisfy uses up the retry budget,
  and then the run raises `UnexpectedModelBehavior`.